In [40]:
import pandas as pd
import numpy as np

# 모델
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

# 전처리
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 평가
from sklearn.metrics import roc_auc_score

# 불균형 처리
from imblearn.over_sampling import SMOTE

# -------------------------------
# 1. 데이터 로드
# -------------------------------
df = pd.read_csv(r"C:\Users\Admin\Documents\GitHub\Horse\data_preprocessing\merged_data_kr_Nan.csv")

# -------------------------------
# 2. 타겟 생성 (1등 여부)
# -------------------------------
df["target"] = (df["순위"] == 1).astype(int)

# -------------------------------
# 3. 불필요 컬럼 제거
# -------------------------------
# ❗ 누수 컬럼 제거
y = (df["순위"] == 1).astype(int)  # 이건 drop 전에 만들어야 함
df = df.drop(columns=[
    "순위",
    "순위점수",
    "경주기록"
], errors="ignore")

# 타겟은 따로 보관


drop_cols = ["마명", "소유자명", "부마명"]  # 텍스트 노이즈
df = df.drop(columns=drop_cols, errors="ignore")

# -------------------------------
# 4. 날짜 처리
# -------------------------------
df["경주일자"] = pd.to_datetime(df["경주일자"])
df["year"] = df["경주일자"].dt.year
df["month"] = df["경주일자"].dt.month

# -------------------------------
# 5. 범주형 인코딩
# -------------------------------
cat_cols = df.select_dtypes(include="object").columns

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = df[col].astype(str)
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le

# -------------------------------
# 6. 결측치 처리
# -------------------------------
df = df.fillna(-1)

# -------------------------------
# 7. Feature / Target 분리
# -------------------------------
X = df.drop(columns=["target", "순위", "경주일자"], errors="ignore")
y = df["target"]

# -------------------------------
# 8. Train/Test 분리
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)



# -------------------------------
# 9. SMOTE 적용 (훈련 데이터만!)
# -------------------------------
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

# -------------------------------
# 10. 모델 정의
# -------------------------------
lgb_model = LGBMClassifier(n_estimators=300, learning_rate=0.05)
xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, use_label_encoder=False, eval_metric='logloss')
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)

# -------------------------------
# 11. 학습
# -------------------------------
lgb_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

# -------------------------------
# 12. 예측 (확률)
# -------------------------------
lgb_pred = lgb_model.predict_proba(X_test)[:, 1]
xgb_pred = xgb_model.predict_proba(X_test)[:, 1]
rf_pred = rf_model.predict_proba(X_test)[:, 1]

# -------------------------------
# 13. 앙상블 (평균)
# -------------------------------
final_pred = (lgb_pred + xgb_pred + rf_pred) / 3

# -------------------------------
# 14. 평가 (ROC-AUC)
# -------------------------------
lgb_auc = roc_auc_score(y_test, lgb_pred)
xgb_auc = roc_auc_score(y_test, xgb_pred)
rf_auc = roc_auc_score(y_test, rf_pred)
final_auc = roc_auc_score(y_test, final_pred)

print("LGBM AUC:", lgb_auc)
print("XGB AUC:", xgb_auc)
print("RF AUC:", rf_auc)
print("ENSEMBLE AUC:", final_auc)

C:\Users\Admin\AppData\Local\Temp\ipykernel_11200\1120753450.py:56: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns


[LightGBM] [Info] Number of positive: 10788, number of negative: 10788
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000848 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2028
[LightGBM] [Info] Number of data points in the train set: 21576, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


c:\Users\Admin\Documents\GitHub\Horse\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:11:10] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


LGBM AUC: 0.6372687890436162
XGB AUC: 0.6396075339218916
RF AUC: 0.6081515795797486
ENSEMBLE AUC: 0.6351755114684389
